# Create control tables and optionally seed examples

In [ ]:
%%configure
{
  "defaultLakehouse": {
    "name": {"variableName": "$(/**/vl_ingestion/lakehouse_name)"},
    "id": {"variableName": "$(/**/vl_ingestion/lakehouse_id)"},
    "workspaceId": {"variableName": "$(/**/vl_ingestion/workspace_id)"}
  }
}

In [ ]:
expected_environment = "dev"
seed_examples = False

In [ ]:
import json
import notebookutils

library = notebookutils.variableLibrary.getLibrary("vl_ingestion")
environment = library.getVariable("environment")
if environment != expected_environment or environment not in {"dev", "uat", "prod"}:
    raise ValueError("Unexpected environment")

for statement in 'CREATE SCHEMA IF NOT EXISTS ops;\n\nCREATE TABLE IF NOT EXISTS ops.connection (\n  connection_id STRING, environment STRING, connector STRING, options_json STRING\n) USING DELTA;\n\nCREATE TABLE IF NOT EXISTS ops.dataset (\n  dataset_id STRING, connection_id STRING, load_mode STRING, enabled BOOLEAN,\n  run_group STRING, options_json STRING\n) USING DELTA;\n\nCREATE TABLE IF NOT EXISTS ops.landing_event (\n  client STRING, environment STRING, dataset_id STRING, batch_id STRING,\n  attempt_id STRING, status STRING, plan_json STRING, manifest_path STRING,\n  object_count BIGINT, error_type STRING, recorded_utc TIMESTAMP\n) USING DELTA;\n'.split(";"):
    if statement.strip():
        spark.sql(statement)

if seed_examples:
    if environment != "dev":
        raise ValueError("Seed examples in DEV only")
    examples = json.loads('{\n  "connections": [\n    {"connection_id":"crm","environment":"dev","connector":"api","options":{"base_url":"https://crm.example.com","auth":"oauth_client_credentials","token_url":"https://identity.example.com/oauth2/token","client_id":"replace-me","client_secret":"crm-client-secret"}},\n    {"connection_id":"workday","environment":"dev","connector":"raas","options":{"base_url":"https://workday.example.com","auth":"basic","user":"replace-me","password_secret":"workday-isu-password"}},\n    {"connection_id":"erp","environment":"dev","connector":"sqlserver","options":{"url":"jdbc:sqlserver://sql.example.com:1433;databaseName=ERP;encrypt=true;trustServerCertificate=false","user":"fabric_reader","password_secret":"erp-reader-password"}},\n    {"connection_id":"partner","environment":"dev","connector":"adls","options":{"root":"abfss://incoming@account.dfs.core.windows.net/deliveries"}}\n  ],\n  "datasets": [\n    {"dataset_id":"crm__accounts","connection_id":"crm","load_mode":"watermark","enabled":false,"run_group":"hourly","options":{"path":"v1/accounts","pagination":"next_link","next_field":"links.next","from_param":"updated_from","to_param":"updated_before","initial_watermark":"2026-01-01T00:00:00Z","lookback_seconds":300,"safety_lag_seconds":120}},\n    {"dataset_id":"workday__workers","connection_id":"workday","load_mode":"full","enabled":false,"run_group":"daily","options":{"path":"ccx/service/customreport2/tenant/owner/Workers","params":{"format":"xml"},"extension":"xml"}},\n    {"dataset_id":"erp__customers","connection_id":"erp","load_mode":"watermark","enabled":false,"run_group":"hourly","options":{"table":"dbo.Customers","watermark_column":"updated_utc","initial_watermark":"2026-01-01T00:00:00Z","safety_lag_seconds":60,"lookback_seconds":300}},\n    {"dataset_id":"erp__orders_changes","connection_id":"erp","load_mode":"sqlserver_cdc","enabled":false,"run_group":"hourly","options":{"capture_instance":"dbo_Orders","initial_lsn":"REPLACE_WITH_BASELINE_LSN_HEX"}},\n    {"dataset_id":"partner__orders","connection_id":"partner","load_mode":"file_incremental","enabled":false,"run_group":"daily","options":{"prefix":"orders","glob":"*.xlsx","max_files":500}}\n  ]\n}\n')
    connections = [(row["connection_id"], row["environment"], row["connector"],
                    json.dumps(row["options"])) for row in examples["connections"]]
    datasets = [(row["dataset_id"], row["connection_id"], row["load_mode"],
                 row["enabled"], row["run_group"], json.dumps(row["options"]))
                for row in examples["datasets"]]
    spark.createDataFrame(connections, "connection_id string, environment string, connector string, options_json string").createOrReplaceTempView("seed_connections")
    spark.createDataFrame(datasets, "dataset_id string, connection_id string, load_mode string, enabled boolean, run_group string, options_json string").createOrReplaceTempView("seed_datasets")
    spark.sql('''MERGE INTO ops.connection t USING seed_connections s
                 ON t.connection_id = s.connection_id AND t.environment = s.environment
                 WHEN NOT MATCHED THEN INSERT *''')
    spark.sql('''MERGE INTO ops.dataset t USING seed_datasets s
                 ON t.dataset_id = s.dataset_id WHEN NOT MATCHED THEN INSERT *''')